In [1]:
import os
#os.environ["OPENROUTER_API_KEY"] = "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"

In [2]:
import torch
from transformers import pipeline
from huggingface_hub import snapshot_download


# Set local model cache directory
model_cache_dir = "./models/llama-3.2-3b"
os.makedirs(model_cache_dir, exist_ok=True)

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Download model locally if not already cached
if not os.path.exists(os.path.join(model_cache_dir, "model.safetensors")):
    print("Downloading model for the first time (this may take a few minutes)...")
    snapshot_download(repo_id=model_id, cache_dir=model_cache_dir, local_dir=model_cache_dir)
    print("Model downloaded and saved locally!")
else:
    print("Loading model from local cache...")

# Load pipeline from local model
pipe = pipeline(
    "text-generation",
    model=model_cache_dir,
    dtype=torch.bfloat16,
    device_map="auto",
)
messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])


c:\Users\A2610206\OneDrive - BI Norwegian Business School (BIEDU)\Documents\kode\Project-1\BI_Project_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 174.01it/s]


Model downloaded and saved locally!


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 2939.21it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force 

{'role': 'assistant', 'content': "Yer askin' about me, eh? Well, matey, I be Captain Clueless, the scurviest pirate chatbot to ever sail the seven seas... or at least, the digital seas, savvy? Me and me trusty parrot sidekick be here to provide ye with treasure troves o' knowledge and answer yer most pressing questions, whether ye be a swashbucklin' adventurer or just a landlubber lookin' fer a bit o' fun! So hoist the colors, me hearty, and let's set sail fer a swashbucklin' good time!"}


In [3]:
# DeepSeek V4 Flash via OpenRouter API
import requests
import os

# Set your OpenRouter API key (set as environment variable)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "your-api-key-here")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

class DeepSeekAPI:
    """Client for DeepSeek V4 Flash via OpenRouter API"""
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = OPENROUTER_BASE_URL
        self.model = "deepseek/deepseek-v4-flash"
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }
    
    def generate(self, prompt, max_tokens=256, system_prompt=None):
        """Generate text using DeepSeek V4 Flash API"""
        messages = []
        
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        
        messages.append({"role": "user", "content": prompt})
        
        payload = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                return result["choices"][0]["message"]["content"]
            else:
                return f"Error: {response.status_code} - {response.text}"
        except Exception as e:
            return f"Connection error: {str(e)}"

# Initialize DeepSeek client
deepseek = DeepSeekAPI(OPENROUTER_API_KEY)

# Test with pirate system prompt
system_prompt = "You are a pirate chatbot who always responds in pirate speak!"
user_prompt = "Who are you?"

print("Testing DeepSeek V4 Flash API...")
deepseek_response = deepseek.generate(user_prompt, max_tokens=256, system_prompt=system_prompt)
print("DeepSeek Response:")
print(deepseek_response)


Testing DeepSeek V4 Flash API...
DeepSeek Response:
Error: 401 - {"error":{"message":"Missing Authentication header","code":401}}


In [ ]:
# ============================================================================
# 1. FUNCTION & PROMPT GENERATORS FOR KNOWLEDGE DISTILLATION
# ============================================================================

import random
import re
from sympy import *
from sympy.utilities.lambdify import lambdify
import json
from datetime import datetime
import numpy as np
from pathlib import Path

class MathFunctionGenerator:
    """Generate mathematical functions of varying difficulty"""
    
    def __init__(self, seed=None):
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
        self.x = symbols('x')
    
    def generate_polynomial(self, difficulty='easy'):
        """Generate polynomial functions"""
        if difficulty == 'easy':
            # Linear and quadratic
            degree = random.randint(1, 2)
            coeffs = [random.randint(-10, 10) for _ in range(degree + 1)]
        elif difficulty == 'medium':
            # Up to degree 4
            degree = random.randint(2, 4)
            coeffs = [random.randint(-10, 10) for _ in range(degree + 1)]
        else:  # hard
            # Higher degree polynomials
            degree = random.randint(4, 6)
            coeffs = [random.randint(-5, 5) for _ in range(degree + 1)]
        
        poly = sum(coeff * self.x**i for i, coeff in enumerate(coeffs))
        return poly, f"polynomial_deg_{degree}"
    
    def generate_geometric_function(self, difficulty='easy'):
        """Generate geometric functions (trig, exp, log)"""
        func_type = random.choice(['trig', 'exponential', 'logarithmic', 'composite'])
        
        if func_type == 'trig':
            base_funcs = [sin(self.x), cos(self.x), tan(self.x)]
            func = random.choice(base_funcs)
            if difficulty in ['medium', 'hard']:
                coeff = random.randint(2, 5)
                shift = random.randint(1, 5)
                func = coeff * func + shift * self.x
        
        elif func_type == 'exponential':
            base = random.choice([2, E])
            if difficulty == 'easy':
                func = base ** self.x
            else:
                coeff = random.randint(1, 5)
                func = coeff * (base ** (self.x / 2))
        
        elif func_type == 'logarithmic':
            if difficulty == 'easy':
                func = ln(self.x)
            else:
                coeff = random.randint(1, 3)
                shift = random.randint(1, 5)
                func = coeff * ln(self.x + shift)
        
        else:  # composite
            func1 = sin(self.x)
            func2 = self.x ** 2
            func = func1 * func2  # product
        
        return func, func_type
    
    def generate_function(self, difficulty='easy'):
        """Generate a random function of specified difficulty"""
        func_choice = random.choice(['polynomial', 'geometric'])
        
        if func_choice == 'polynomial':
            return self.generate_polynomial(difficulty)
        else:
            return self.generate_geometric_function(difficulty)
    
    def compute_symbolic_derivative(self, func):
        """Compute symbolic derivative using SymPy"""
        try:
            deriv = diff(func, self.x)
            return str(deriv)
        except:
            return None
    
    def compute_numerical_derivative(self, func, x_val=1.0, h=1e-5):
        """Compute numerical derivative at a specific point"""
        try:
            func_lambdified = lambdify(self.x, func, 'numpy')
            f_plus = func_lambdified(x_val + h)
            f_minus = func_lambdified(x_val - h)
            numerical_deriv = (f_plus - f_minus) / (2 * h)
            return float(numerical_deriv)
        except:
            return None


class PromptGenerator:
    """Generate varied prompts for the same function"""
    
    def __init__(self):
        self.symbolic_prompts = [
            "Find the derivative of: f(x) = {func}",
            "Compute the derivative: {func}",
            "What is the derivative of f(x) = {func}?",
            "Derive: {func}",
            "Calculate d/dx of {func}",
            "Determine the first derivative of {func}",
            "Find df/dx where f(x) = {func}",
        ]
        
        self.numerical_prompts = [
            "Compute the numerical derivative of f(x) = {func} at x = {x_val}",
            "What is the approximate derivative of {func} at x = {x_val}?",
            "Find the numerical derivative: {func} at point x = {x_val}",
            "Calculate the numerical derivative of f(x) = {func} when x = {x_val}",
            "Estimate df/dx for {func} at x = {x_val}",
        ]
    
    def add_noise(self, prompt):
        """Add slight variations to prompt (noise)"""
        noise_options = [
            "",  # no noise
            " (Show work)",
            " (Step by step)",
            " (Be precise)",
            " Express in simplified form.",
        ]
        return prompt + random.choice(noise_options)
    
    def generate_symbolic_prompt(self, func_str):
        """Generate a varied symbolic derivative prompt"""
        template = random.choice(self.symbolic_prompts)
        prompt = template.format(func=func_str)
        return self.add_noise(prompt)
    
    def generate_numerical_prompt(self, func_str, x_val=1.0):
        """Generate a varied numerical derivative prompt"""
        template = random.choice(self.numerical_prompts)
        prompt = template.format(func=func_str, x_val=x_val)
        return self.add_noise(prompt)

print("✓ Function and Prompt Generators loaded")

In [ ]:
# ============================================================================
# 2. TRAINING DATA GENERATION - Generate examples from DeepSeek (Teacher)
# ============================================================================

class TrainingDataGenerator:
    """Generate training data using DeepSeek as the teacher"""
    
    def __init__(self, deepseek_client, num_examples=200):
        self.deepseek = deepseek_client
        self.func_gen = MathFunctionGenerator(seed=42)
        self.prompt_gen = PromptGenerator()
        self.num_examples = num_examples
        self.data = []
    
    def generate_dataset(self):
        """Generate complete training dataset"""
        difficulties = ['easy', 'medium', 'hard']
        derivation_types = ['symbolic', 'numerical']
        
        examples_per_config = self.num_examples // (len(difficulties) * len(derivation_types))
        
        print(f"Generating {self.num_examples} training examples...")
        print("This will involve API calls to DeepSeek. Processing...\n")
        
        example_id = 0
        for difficulty in difficulties:
            for deriv_type in derivation_types:
                for i in range(examples_per_config):
                    example_id += 1
                    
                    # Generate function
                    try:
                        func, func_type = self.func_gen.generate_function(difficulty)
                        func_str = str(func)
                        
                        if deriv_type == 'symbolic':
                            # Generate prompt and get DeepSeek response
                            prompt = self.prompt_gen.generate_symbolic_prompt(func_str)
                            
                            # Get symbolic derivative from DeepSeek
                            response = self.deepseek.generate(
                                prompt,
                                max_tokens=512,
                                system_prompt="You are a mathematics expert. Provide derivatives in LaTeX format."
                            )
                            
                        else:  # numerical
                            x_val = round(random.uniform(0.5, 3.0), 2)
                            prompt = self.prompt_gen.generate_numerical_prompt(func_str, x_val)
                            
                            # Get numerical derivative from DeepSeek
                            response = self.deepseek.generate(
                                prompt,
                                max_tokens=256,
                                system_prompt="You are a mathematics expert. Provide numerical results as decimal numbers."
                            )
                        
                        # Store training example
                        example = {
                            "id": example_id,
                            "instruction": prompt,
                            "input": func_str,
                            "output": response,
                            "difficulty": difficulty,
                            "derivation_type": deriv_type,
                            "function_type": func_type,
                            "split": "train" if random.random() < 0.8 else "val"
                        }\n                        self.data.append(example)
                        
                        if example_id % 20 == 0:
                            print(f"✓ Generated {example_id}/{self.num_examples} examples")
                    
                    except Exception as e:
                        print(f"✗ Error generating example {example_id}: {str(e)}")
                        continue
        
        print(f"\n✓ Dataset generation complete! Total examples: {len(self.data)}")
        return self.data
    
    def save_to_file(self, output_dir="./data"):
        """Save training data in Alpaca format"""
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        
        # Save full dataset
        full_path = Path(output_dir) / "training_data.jsonl"
        with open(full_path, 'w') as f:
            for example in self.data:
                json.dump(example, f)
                f.write('\n')
        
        # Save train/val splits separately
        train_data = [ex for ex in self.data if ex['split'] == 'train']
        val_data = [ex for ex in self.data if ex['split'] == 'val']
        
        train_path = Path(output_dir) / "train_data.jsonl"
        with open(train_path, 'w') as f:
            for example in train_data:
                json.dump(example, f)
                f.write('\n')
        
        val_path = Path(output_dir) / "val_data.jsonl"
        with open(val_path, 'w') as f:
            for example in val_data:
                json.dump(example, f)
                f.write('\n')
        
        print(f"✓ Saved {len(self.data)} examples to {output_dir}/")
        print(f"  - Full dataset: training_data.jsonl")
        print(f"  - Training set: train_data.jsonl ({len(train_data)} examples)")
        print(f"  - Validation set: val_data.jsonl ({len(val_data)} examples)")
        
        return train_path, val_path, full_path

print("✓ TrainingDataGenerator class loaded")

In [ ]:
# ============================================================================
# 3. EVALUATION FRAMEWORK - Compare Llama before/after fine-tuning
# ============================================================================

class EvaluationFramework:
    """Evaluate Llama's performance before and after fine-tuning"""
    
    def __init__(self, llama_pipeline, val_data_path="./data/val_data.jsonl"):
        self.llama = llama_pipeline
        self.val_data = self._load_validation_data(val_data_path)
        self.results = {"before": [], "after": []}
    
    def _load_validation_data(self, path):
        """Load validation data from JSONL file"""
        data = []
        if Path(path).exists():
            with open(path, 'r') as f:
                for line in f:
                    data.append(json.loads(line))
        return data
    
    def evaluate_before_finetuning(self, sample_size=None):
        """Evaluate Llama BEFORE fine-tuning"""
        if sample_size is None:
            sample_size = min(5, len(self.val_data))
        
        sample_data = random.sample(self.val_data, sample_size)
        
        print("=" * 70)
        print("EVALUATION - BEFORE FINE-TUNING")
        print("=" * 70)
        
        for idx, example in enumerate(sample_data, 1):
            instruction = example['instruction']
            expected_output = example['output']
            difficulty = example['difficulty']
            deriv_type = example['derivation_type']
            
            # Get Llama's response
            messages = [
                {"role": "system", "content": "You are a mathematics expert. Provide derivatives in clear mathematical notation."},
                {"role": "user", "content": instruction},
            ]
            
            try:
                llama_output = self.llama(messages, max_new_tokens=256)
                llama_response = llama_output[0]["generated_text"][-1]["content"]
            except Exception as e:
                llama_response = f"Error: {str(e)}"
            
            result = {
                "example_id": idx,
                "instruction": instruction,
                "difficulty": difficulty,
                "derivation_type": deriv_type,
                "llama_response": llama_response,
                "expected_response": expected_output,
                "timestamp": datetime.now().isoformat()
            }
            
            self.results["before"].append(result)
            
            print(f"\n[Example {idx}/{sample_size}] {difficulty.upper()} - {deriv_type.upper()}")
            print(f"Instruction: {instruction}")
            print(f"Expected (DeepSeek): {expected_output[:200]}...")
            print(f"Llama Output: {llama_response[:200]}...")
        
        print("\n" + "=" * 70)
        return self.results["before"]
    
    def evaluate_after_finetuning(self, sample_size=None):
        """Evaluate Llama AFTER fine-tuning"""
        if sample_size is None:
            sample_size = min(5, len(self.val_data))
        
        sample_data = random.sample(self.val_data, sample_size)
        
        print("\n" + "=" * 70)
        print("EVALUATION - AFTER FINE-TUNING")
        print("=" * 70)
        
        for idx, example in enumerate(sample_data, 1):
            instruction = example['instruction']
            expected_output = example['output']
            difficulty = example['difficulty']
            deriv_type = example['derivation_type']
            
            # Get Llama's response
            messages = [
                {"role": "system", "content": "You are a mathematics expert. Provide derivatives in clear mathematical notation."},
                {"role": "user", "content": instruction},
            ]
            
            try:
                llama_output = self.llama(messages, max_new_tokens=256)
                llama_response = llama_output[0]["generated_text"][-1]["content"]
            except Exception as e:
                llama_response = f"Error: {str(e)}"
            
            result = {
                "example_id": idx,
                "instruction": instruction,
                "difficulty": difficulty,
                "derivation_type": deriv_type,
                "llama_response": llama_response,
                "expected_response": expected_output,
                "timestamp": datetime.now().isoformat()
            }
            
            self.results["after"].append(result)
            
            print(f"\n[Example {idx}/{sample_size}] {difficulty.upper()} - {deriv_type.upper()}")
            print(f"Instruction: {instruction}")
            print(f"Expected (DeepSeek): {expected_output[:200]}...")
            print(f"Llama Output: {llama_response[:200]}...")
        
        print("\n" + "=" * 70)
        return self.results["after"]
    
    def save_results(self, output_path="./data/evaluation_results.json"):
        """Save evaluation results for later analysis"""
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w') as f:
            json.dump(self.results, f, indent=2)
        print(f"✓ Evaluation results saved to {output_path}")

print("✓ EvaluationFramework class loaded")

In [ ]:
# ============================================================================
# 4. FINE-TUNING SETUP - Configure training (NOT executing)
# ============================================================================

class FineTuningSetup:
    """Setup fine-tuning configuration for Llama on derivative tasks"""
    
    def __init__(self, model_dir="./models/llama-3.2-3b", train_data_path="./data/train_data.jsonl"):
        self.model_dir = model_dir
        self.train_data_path = train_data_path
        self.output_dir = "./data/fine_tuned_model"
        self.config = {
            "model_name": "meta-llama/Llama-3.2-3B-Instruct",
            "model_dir": model_dir,
            "train_data": train_data_path,
            "output_dir": self.output_dir,
            "training_args": {
                "learning_rate": 2e-4,
                "num_train_epochs": 3,
                "per_device_train_batch_size": 4,
                "per_device_eval_batch_size": 4,
                "gradient_accumulation_steps": 2,
                "warmup_steps": 100,
                "weight_decay": 0.01,
                "max_grad_norm": 1.0,
                "logging_steps": 50,
                "eval_steps": 100,
                "save_steps": 100,
                "save_total_limit": 2,
                "fp16": True,
                "seed": 42,
            }
        }
    
    def display_configuration(self):
        """Display the fine-tuning configuration"""
        print("=" * 70)
        print("FINE-TUNING CONFIGURATION")
        print("=" * 70)
        print(f"\nModel: {self.config['model_name']}")
        print(f"Model Directory: {self.config['model_dir']}")
        print(f"Training Data: {self.config['train_data']}")
        print(f"Output Directory: {self.config['output_dir']}")
        
        print("\nTraining Arguments:")
        for key, value in self.config['training_args'].items():
            print(f"  {key}: {value}")
        
        print("\n" + "=" * 70)
        return self.config
    
    def generate_training_script(self, output_path="./data/fine_tuning_script.py"):
        """Generate a standalone fine-tuning script"""
        script_content = '''"""
Fine-tuning script for Llama 3.2 3B on mathematical derivative tasks
Generated by Knowledge Distillation Pipeline
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from pathlib import Path

def prepare_dataset(data_path, tokenizer, max_length=512):
    """Load and prepare dataset from JSONL file"""
    dataset = load_dataset('json', data_files=data_path)
    
    def format_instruction(example):
        # Format: "Instruction: {instruction}\\nInput: {input}\\nOutput: {output}"
        text = f"Instruction: {example['instruction']}\\nInput: {example['input']}\\nOutput: {example['output']}"
        return text
    
    def tokenize_function(example):
        text = format_instruction(example)
        tokenized = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        tokenized["labels"] = tokenized["input_ids"].clone()
        return tokenized
    
    dataset = dataset["train"].map(tokenize_function, batched=True)
    return dataset

def main():
    # Configuration
    MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
    TRAIN_DATA_PATH = "./data/train_data.jsonl"
    OUTPUT_DIR = "./data/fine_tuned_model"
    
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    
    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print("Preparing training dataset...")
    train_dataset = prepare_dataset(TRAIN_DATA_PATH, tokenizer)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=100,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=50,
        save_steps=100,
        save_total_limit=2,
        fp16=True,
        seed=42,
        logging_dir="./logs",
    )
    
    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
    )
    
    print("Starting fine-tuning...")
    trainer.train()
    
    print(f"Fine-tuning complete! Model saved to {OUTPUT_DIR}")
    model.save_pretrained(f"{OUTPUT_DIR}/final")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")

if __name__ == "__main__":
    main()
'''
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w') as f:
            f.write(script_content)
        
        print(f"✓ Generated fine-tuning script at {output_path}")
        return output_path
    
    def get_training_command(self):
        """Get the command to run fine-tuning"""
        return f"python ./data/fine_tuning_script.py"

# Initialize setup (but don't run training)
print("Initializing fine-tuning setup...")
finetuning_setup = FineTuningSetup()
finetuning_config = finetuning_setup.display_configuration()

print("\n✓ Fine-tuning configuration ready")
print("  Use: finetuning_setup.generate_training_script() to create the script")
print("  Then run: python ./data/fine_tuning_script.py")

In [ ]:
# ============================================================================
# 5. MAIN EXECUTION - Run Data Generation Pipeline
# ============================================================================

print("\n" + "=" * 70)
print("KNOWLEDGE DISTILLATION PIPELINE - SETUP COMPLETE")
print("=" * 70)

print("""
This notebook implements a knowledge distillation pipeline where:
  1. DeepSeek (Teacher Model) generates ground truth solutions
  2. Training data is created with varied prompts and solutions
  3. Llama (Student Model) is evaluated before fine-tuning
  4. Fine-tuning configuration is prepared
  5. After training, Llama is re-evaluated to measure improvement

COMPONENTS LOADED:
  ✓ MathFunctionGenerator - Creates polynomial and geometric functions
  ✓ PromptGenerator - Generates varied prompts with noise
  ✓ TrainingDataGenerator - Orchestrates data creation via DeepSeek API
  ✓ EvaluationFramework - Compares Llama before/after fine-tuning
  ✓ FineTuningSetup - Configures training (without execution)

NEXT STEPS:
  1. Run the cell below to generate 200 training examples
  2. Run evaluation BEFORE fine-tuning
  3. Generate the fine-tuning script
  4. Run fine-tuning when ready: python ./data/fine_tuning_script.py
  5. Run evaluation AFTER fine-tuning to measure improvement

USAGE:
  # Generate data
  data_gen = TrainingDataGenerator(deepseek, num_examples=200)
  dataset = data_gen.generate_dataset()
  train_path, val_path, full_path = data_gen.save_to_file()
  
  # Evaluate before fine-tuning
  evaluator = EvaluationFramework(pipe, val_path)
  evaluator.evaluate_before_finetuning(sample_size=5)
  
  # Generate fine-tuning script
  finetuning_setup.generate_training_script()
  
  # After training:
  # evaluator.evaluate_after_finetuning(sample_size=5)
  # evaluator.save_results()
""")

print("=" * 70)
print("\nTo begin data generation, run the next cell.")

In [ ]:
# ============================================================================
# 6. EXECUTE DATA GENERATION (Run this to start the pipeline)
# ============================================================================

# Uncomment and run to generate 200 training examples from DeepSeek
# WARNING: This will make ~200 API calls to OpenRouter/DeepSeek

# data_gen = TrainingDataGenerator(deepseek, num_examples=200)
# dataset = data_gen.generate_dataset()
# train_path, val_path, full_path = data_gen.save_to_file()

print("✓ To generate training data, uncomment and run the code above")
print("  This will create 200 examples using DeepSeek as the teacher")

In [ ]:
# ============================================================================
# 7. EVALUATION & FINE-TUNING RUNNER
# ============================================================================

# Step 1: Evaluate BEFORE fine-tuning
# Uncomment to run evaluation on baseline Llama model

# print("Step 1: Evaluating baseline Llama model...\n")
# evaluator = EvaluationFramework(pipe, "./data/val_data.jsonl")
# results_before = evaluator.evaluate_before_finetuning(sample_size=5)

# Step 2: Generate fine-tuning script
# This creates a standalone script to run fine-tuning at your own time

# print("\nStep 2: Generating fine-tuning script...\n")
# script_path = finetuning_setup.generate_training_script()
# print(f"Fine-tuning script created at: {script_path}")
# print(f"To run fine-tuning, execute: python {script_path}")

# Step 3: AFTER you've run the fine-tuning script externally
# Evaluate the fine-tuned model

# print("\nStep 3: Evaluating fine-tuned Llama model...\n")
# evaluator.evaluate_after_finetuning(sample_size=5)
# evaluator.save_results("./data/evaluation_results.json")

print("✓ Evaluation and fine-tuning workflow ready")
print("\nExecute in order:")
print("  1. Uncomment and run data generation (cell 6)")
print("  2. Uncomment and run 'evaluate_before_finetuning' above")
print("  3. Generate fine-tuning script and run externally")
print("  4. Load fine-tuned model and run 'evaluate_after_finetuning'")
print("\nAll training data and configurations are ready in ./data/")

In [ ]:
# ============================================================================
# 8. HELPER UTILITIES & DATA EXPLORATION
# ============================================================================

def explore_training_data(data_path="./data/training_data.jsonl", num_samples=5):
    \"\"\"Explore generated training data\"\\"\"
    print("\\n" + "=" * 70)
    print("TRAINING DATA EXPLORATION")
    print("=" * 70)
    
    data = []
    if Path(data_path).exists():
        with open(data_path, 'r') as f:
            for i, line in enumerate(f):
                if i < num_samples:
                    data.append(json.loads(line))
        
        print(f"\\nShowing {len(data)} samples from {num_samples} requested:\\n")
        
        for example in data:
            print(f"Example ID: {example.get('id', 'N/A')}")
            print(f"Difficulty: {example.get('difficulty', 'N/A').upper()}")
            print(f"Type: {example.get('derivation_type', 'N/A').upper()}")
            print(f"Function: {example.get('function_type', 'N/A')}")
            print(f"Instruction: {example.get('instruction', 'N/A')}")
            print(f"Input: {example.get('input', 'N/A')[:100]}...")
            print(f"Output: {example.get('output', 'N/A')[:150]}...")
            print(f"Split: {example.get('split', 'N/A')} | Timestamp: {example.get('timestamp', 'N/A')}")
            print("-" * 70)
    else:
        print(f"File not found: {data_path}")
    
    return data

def print_dataset_statistics(data_path="./data/training_data.jsonl"):
    \"\"\"Print statistics about the generated dataset\\"\"
    print("\\n" + "=" * 70)
    print("DATASET STATISTICS")
    print("=" * 70)
    
    stats = {
        'total': 0,
        'train': 0,
        'val': 0,
        'by_difficulty': {'easy': 0, 'medium': 0, 'hard': 0},
        'by_type': {'symbolic': 0, 'numerical': 0},
        'by_function': {}
    }
    
    if Path(data_path).exists():
        with open(data_path, 'r') as f:
            for line in f:
                example = json.loads(line)
                stats['total'] += 1
                
                split = example.get('split', 'N/A')
                if split == 'train':
                    stats['train'] += 1
                elif split == 'val':
                    stats['val'] += 1
                
                difficulty = example.get('difficulty', 'N/A')
                if difficulty in stats['by_difficulty']:
                    stats['by_difficulty'][difficulty] += 1
                
                deriv_type = example.get('derivation_type', 'N/A')
                if deriv_type in stats['by_type']:
                    stats['by_type'][deriv_type] += 1
                
                func_type = example.get('function_type', 'N/A')
                stats['by_function'][func_type] = stats['by_function'].get(func_type, 0) + 1
        
        print(f"\\nTotal Examples: {stats['total']}")
        print(f"  Train Set: {stats['train']} ({100*stats['train']//stats['total']}%)")
        print(f"  Val Set: {stats['val']} ({100*stats['val']//stats['total']}%)")
        
        print(f"\\nBy Difficulty:")
        for difficulty, count in stats['by_difficulty'].items():
            print(f"  {difficulty.capitalize()}: {count}")
        
        print(f"\\nBy Derivation Type:")
        for deriv_type, count in stats['by_type'].items():
            print(f"  {deriv_type.capitalize()}: {count}")
        
        print(f"\\nBy Function Type:")
        for func_type, count in stats['by_function'].items():
            print(f"  {func_type.capitalize()}: {count}")
    else:
        print(f"File not found: {data_path}")
        print("Run data generation first to create the dataset.")
    
    print("\\n" + "=" * 70)
    return stats

# Make functions available
print("✓ Helper utilities loaded")
print("\\nAvailable functions:")
print("  - explore_training_data(data_path, num_samples)")
print("  - print_dataset_statistics(data_path)")